In [89]:
import numpy as np
import pandas as pd

# Cost Functions and Unit Selection

From the target phonemes, a path needs to be found through the state transition network (sequence of database units), that minimizes a cost function.

![image.png](resources/UnitSelectionCosts.png)

## Target Cost

The target cost is the weighted sum of the differences between elements of the target and candidate feature vectors.

$$ C^{t}(t_i, u_i) = \sum_{j=1}^{p} w_j C^{t}_{j}(t_i, u_i) $$

Where $p$ ranges from 20 to 30 and is the number of sub-costs, or the weighted differences between the feature vectors.

## Concatenation Cost

Determined by the weight of $q$ concatenation sub-costs.

$$ C^c_{(u_{i-1}, u_i)} = \sum_{j=1}^q w^c_j C_j^{c}(u_{i-1}, u_i) $$

## Final Cost Function

$$ C(t_1^n, u_1^n) = \sum_{i=1}^{n} \sum_{j=1}^{p} w_j^tC_j^t(t_i, u_i^t) + \sum_{i=2}^{n} \sum_{j=1}^{q} w_j^cC_j^c(u_j, u_i) + C^s(S, u_1) + C^c(u_n, S) $$

Where $S$ is silence.

The units are then selected in order to minimize this cost function via Viterbi search.

$$ \bar{u}_1^n = \min_{u_1,\ldots,u_n} \mathcal{C}(t_1^n,u_1^n) $$

# Cost Function Implementation

Target and concatenation costs require summing the weighted sub-costs across available features. The paper states that sub-cost functions use a comparison function that returns a result in range [0, 1] (1 being the same).

## Sub-Cost Computation

We have two distinct types of features available - phonetic identity and acoustic. Phonetic features binary options and one-hot encodings. Acoustic features are scalar and vector floats.

### AI Says

Binary options and one-hot encodings can just be matched - outputting a 0 or 1.

Acoustic feature scalars can be normalized then z-score squared differenced. Acoustic vector features can be normalized (or not) then euclidean distance calculated.

In [90]:
# Cost and sub-cost computation

# We have a target feature vector and a candidate feature vector
# We have a dictionary of weights for the costs of each sub-feature
# Phonetic feature costs are binary options that the weight then gets applied to
# Acoustic scalar features are normalized then z-score square differenced then weighted
# Acoustic vector features use euclidean distance then weighted.

ACOUSTIC_WEIGHTS = {
    'log_power' : 0.5,
    'log_pitch' : 0.5,
    'mfcc'  : 0.5,
}

def target_cost(target, candidate, weights):
    cost = 0
    for key, weight in weights.items():
        cost += weight * target[key] * candidate[key]
    return cost

def concatenation_cost(prev, candidate, weights):
    cost = 0
    for key, weight in weights.items():
        cost += weight * prev[key] * candidate[key]
    return cost

# Regression Training

We will need phoneme sets on which to train on. Likely a dictionary of common words, then break those into phonemes should be sufficient.

For each example unit in the database from the set being trained:
1. Example unit is the target unit
- Calculate acoustic difference between target unit and all instances of same phoneme in database with the objective distance function. This is the cepstral distance.
- Use n=20 best matches.
- Determine target sub-costs between target and n-best matches.
2. Collect objective distances and target costs across all targets and their n-best matches.
3. Use linear regression to compute the weights for the target subcosts in the current phoneme set.
4. Repeat for each phoneme set.

## Practical Terms

Practical implementation:
1. Select as a target phoneme sequence a lab file and its corresponding wave file.
- This can be processed and in a database already, but it needs to be partitioned and labelled as training data.
2. For each target phoneme in the sequence:
- Calculate acoustic (cepstral) difference between target phoneme and all instances of phoneme in database.
- Keep top N=20
- Calculate target sub-costs for N candidates.
3. Accumulate sub-cost differences and acoustic spectral distances.

$$\mathbf{A}_{\text{total}} \leftarrow \mathbf{A}_{\text{total}} + \mathbf{X}_{\text{batch}}^T \mathbf{X}_{\text{batch}}$$
$$\mathbf{b}_{\text{total}} \leftarrow \mathbf{b}_{\text{total}} + \mathbf{X}_{\text{batch}}^T \mathbf{Y}_{\text{batch}}$$

4. Repeat steps 1-3.
5. Solve the accumulated matrix/vector for the weights.

$$\mathbf{w}^t = (\mathbf{A}_{\text{total}})^{-1} \mathbf{b}_{\text{total}}$$


In [91]:
# Data loading and massaging

# Batched training
# We need a way to extract a target sequence from a lab file on which to train on
# Features are calculated and thrown in a database.
# If each phoneme feature had a label with: filename, occurrence in file, they could be found in the database.

# Find unique lab file names from id column. Each lab file gives a target sequence.
# Iterate over lab file to extract phonemes id labels.
# Construct and return target feature sequence
def const_target_feat_seq(train_db_file):
    df = pd.read_parquet(train_db_file)

    # Extract lab file name (everything before the last underscore+digits)
    df["lab_file"] = df["id"].str.rsplit("_", n=1).str[0]
    # Extract numeric index so we can sort phonemes in the correct order
    df["ph_idx"] = df["id"].str.rsplit("_", n=1).str[1].astype(int)

    sequences = []
    for lab, grp in df.groupby("lab_file", sort=False):
        grp_sorted = grp.sort_values("ph_idx")
        # Drop the helper columns we added
        grp_sorted = grp_sorted.drop(columns=["lab_file", "ph_idx"])
        sequences.append(grp_sorted)

    return sequences

In [92]:
# Regression Training

def train_on_seq(target_sequence, synthesis_db_file):
    # Feature columns used for target sub-costs (binary comparison)
    SUBCOST_COLS = ['nph', 'pph', 'is_vowel', 'is_voiced', 'manner', 'place']
    p = len(SUBCOST_COLS)
    N_BEST = 20

    # Accumulators for normal equations:  A += X^T X,  b += X^T y
    A_total = np.zeros((p, p))
    b_total = np.zeros(p)

    # Find N=20 best matches for each target phoneme from MFCC distance
    # Build and accumulate target sub-costs
    for idx, target_phoneme in target_sequence.iterrows():
        # Extract MFCC from all matching db phonemes and calculate distance
        matching_phonemes = pd.read_parquet(
            synthesis_db_file,
            filters=[("ph", "==", target_phoneme["ph"])]
        )

        target_mfcc = target_phoneme['mfcc']
        candidate_mfccs = np.array(matching_phonemes['mfcc'].tolist())

        cands = matching_phonemes.assign(
            dist=np.linalg.norm(target_mfcc - candidate_mfccs, axis=1)
        ).sort_values(by='dist', ascending=True).head(N_BEST)

        n = len(cands)
        if n == 0:
            continue

        # Build X: (n x p) binary sub-cost matrix
        # Each entry = 1 if candidate feature matches target feature, else 0
        X_batch = np.zeros((n, p))
        for j, col in enumerate(SUBCOST_COLS):
            X_batch[:, j] = (cands[col].values == target_phoneme[col]).astype(float)

        # y: (n,) vector of acoustic (cepstral) distances
        y_batch = cands['dist'].values

        # Accumulate normal equations
        A_total += X_batch.T @ X_batch
        b_total += X_batch.T @ y_batch

    return A_total, b_total

In [94]:
# Run training

# Debug - one phenome seems to be a perfect match in every set of candidates(?)

SUBCOST_COLS = ['nph', 'pph', 'is_vowel', 'is_voiced', 'manner', 'place']
p = len(SUBCOST_COLS)

# Accumulate A and b over all training sequences
A_total = np.zeros((p, p))
b_total = np.zeros(p)

sequences = const_target_feat_seq('datasets/cmu_us_awb_arctic/feats/a0_train.parquet')
for seq in sequences:
    A, b = train_on_seq(seq, 'datasets/cmu_us_awb_arctic/feats/a0.parquet')
    A_total += A
    b_total += b

# Solve once after all sequences
print(A_total)
print(b_total)
w = np.linalg.solve(A_total, b_total)
print('Learned weights:')
for col, val in zip(SUBCOST_COLS, w):
    print(f'  {col:>12s}: {val:.6f}')

[[11931.  6546. 11931. 11931. 11931. 11931.]
 [ 6546. 20583. 20583. 20583. 20583. 20583.]
 [11931. 20583. 65120. 65120. 65120. 65120.]
 [11931. 20583. 65120. 65120. 65120. 65120.]
 [11931. 20583. 65120. 65120. 65120. 65120.]
 [11931. 20583. 65120. 65120. 65120. 65120.]]
[ 395051.02391102  687049.22804893 2456805.44373145 2456805.44373145
 2456805.44373145 2456805.44373145]


LinAlgError: Singular matrix